In [1]:
import xarray as xr
import pickle
import numpy as np
from datetime import datetime
import time
from tqdm import tqdm

import os
os.environ['WXSYSLIBDIR']='/g/data/gb02/mb0427/WxSysLib'
os.environ['SWTpath']='/home/565/mb0427/gdata-gb02/Australian_synoptic_weather_types/'
import sys
sys.path.append(os.environ['WXSYSLIBDIR'])
sys.path.append(os.environ['SWTpath'])

from assign.assign import assign_grid_to_SWT
from utils.general.date_utils import generate_datetimes,generate_datetimes_months
from utils.general.nci_utils import get_GADI_ERA5_filename


In [20]:
ds=xr.open_dataset(f"{os.environ['SWTpath']}/SWT_fields/SWT_climatology.nc")
ds.close()

for indt in tqdm(generate_datetimes_months(datetime(2025,3,1),datetime(2025,4,1))):
    
    fn = get_GADI_ERA5_filename("u",indt,stream='hourly', level_type='pressure-levels')
    u = xr.open_dataset(fn).sel(level=850,latitude=slice(0,-60),longitude=slice(100,180))
    mask = u.time.dt.hour == 12
    u = u.sel(time=mask)
    u.close()
    
    fn = get_GADI_ERA5_filename("v",indt,stream='hourly', level_type='pressure-levels')
    v = xr.open_dataset(fn).sel(level=850,latitude=slice(0,-60),longitude=slice(100,180))
    mask = v.time.dt.hour == 12
    v = v.sel(time=mask)
    v.close()
    
    for t in u.time:
        SWTlabel=assign_grid_to_SWT(u.sel(time=t),v.sel(time=t),
                                       latname="latitude",lonname="longitude",
                                       cluster_filename=f"{os.environ['SWTpath']}/SWT_fields/SWT_data.nc",
                                       quiet=True)
        new_cluster = xr.DataArray(
            [SWTlabel],
            coords={"time": [t.values]},
            dims=["time"]
        )
        updated_cluster_series = xr.concat([ds["assigned_SWT"], new_cluster], dim="time")
        ds_updated = updated_cluster_series.to_dataset()
        ds_updated = ds_updated.assign_coords(SWTs=ds.SWTs)
        new_attrs = ds.attrs
        new_attrs['last updated']=f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        new_attrs['start month']=f"{ds_updated.time.isel(time=0).dt.strftime('%Y/%m').values}"
        new_attrs['end month']=f"{ds_updated.time.isel(time=-1).dt.strftime('%Y/%m').values}"
        ds_updated = ds_updated.assign_attrs(new_attrs)
        ds=ds_updated.copy()

100%|██████████| 2/2 [00:04<00:00,  2.45s/it]


In [22]:
ds.to_netcdf(f"{os.environ['SWTpath']}/SWT_fields/SWT_climatology.nc")